# Compare best hyperparameters across experiment folders
This notebook loads multiple experiment result folders (as produced by the training runs), extracts the best hyperparameter configuration based on **validation** performance, and summarizes everything in a single comparison table.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys
import pandas as pd

def get_dir_n_levels_up(path, n):
    for _ in range(n):
        path = os.path.dirname(path)
    return path

# Adjust if needed: this mirrors your existing notebook logic
proj_root = get_dir_n_levels_up(os.path.abspath("__file__"), 4)
sys.path.append(proj_root)

from opinion_dynamics.utils.experiment import process_experiment

C:\Users\Chainsword\AppData\Local\Temp\ipykernel_79108\3130451366.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## 1) List the experiment folders you want to compare
These should be subfolders under `./results/` (same as in your original notebook).

In [ ]:
# Put your experiment folders here (subfolders under ./results/)
experiment_sub_dirs = [
    "2025Oct28-115156_configs", # coca 1, seed 42
    "2025Oct29-151336_configs", # coca 2, seed 42
    "2025Oct30-234652_configs", # coca 1, seed 15
    "2025Oct31-115130_configs", # coca 2, seed 15
]

results_root = os.path.join(os.path.abspath("."), "results")
metric = "episode_rewards_mean"  # change if you want to select by a different metric

In [4]:
# Cell 2: best hyperparams + best seed + best epoch/frame (peak validation at any point)
def extract_best_peak_validation(df: pd.DataFrame, metric: str = "episode_rewards_mean"):
    # validation rows only
    df_val = df[df["epoch_type"] == "validation"].copy() if "epoch_type" in df.columns else df.copy()
    if df_val.empty:
        raise ValueError("No validation rows found (epoch_type=='validation').")
    if metric not in df_val.columns:
        raise ValueError(f"Metric '{metric}' not found in columns: {list(df_val.columns)}")

    # hyperparameter columns (as in your original notebook)
    hyperparam_columns = [c for c in df_val.columns if "sub_exp_cfg" in c]

    # pick the single best row across ALL seeds/epochs/frames
    best_row = df_val.sort_values(metric, ascending=False).iloc[0]

    out = {"best_peak_reward": float(best_row[metric])}

    # include identifiers if present
    for col in ["seed", "frame_stamp", "epoch", "sub_experiment_path", "experiment_name"]:
        if col in best_row.index:
            out[f"best_{col}"] = best_row[col]

    # include hyperparams from that best row
    for c in hyperparam_columns:
        out[c] = best_row[c]

    return out


rows = []
for sub_dir in experiment_sub_dirs:
    exp_path = os.path.join(results_root, sub_dir)
    if not os.path.isdir(exp_path):
        print(f"[WARN] Not found: {exp_path}")
        continue

    df = process_experiment(exp_path)
    best = extract_best_peak_validation(df, metric=metric)
    best["experiment_sub_dir"] = sub_dir
    rows.append(best)

compare_df = pd.DataFrame(rows)

if len(compare_df) == 0:
    compare_df
else:
    preferred = [
        "experiment_sub_dir",
        "best_peak_reward",
        "best_seed",
        "best_epoch",
        "best_frame_stamp",
        "best_sub_experiment_path",
        "best_experiment_name",
    ]
    cols = [c for c in preferred if c in compare_df.columns] + [c for c in compare_df.columns if c not in preferred]
    compare_df = compare_df[cols].sort_values("best_peak_reward", ascending=False).reset_index(drop=True)
    compare_df


## 3) Build the comparison table

In [5]:
# Cell 2: best hyperparams + best seed + best epoch/frame (peak validation at any point)
def extract_best_peak_validation(df: pd.DataFrame, metric: str = "episode_rewards_mean"):
    # validation rows only
    df_val = df[df["epoch_type"] == "validation"].copy() if "epoch_type" in df.columns else df.copy()
    if df_val.empty:
        raise ValueError("No validation rows found (epoch_type=='validation').")
    if metric not in df_val.columns:
        raise ValueError(f"Metric '{metric}' not found in columns: {list(df_val.columns)}")

    # hyperparameter columns (as in your original notebook)
    hyperparam_columns = [c for c in df_val.columns if "sub_exp_cfg" in c]

    # pick the single best row across ALL seeds/epochs/frames
    best_row = df_val.sort_values(metric, ascending=False).iloc[0]

    out = {"best_peak_reward": float(best_row[metric])}

    # include identifiers if present
    for col in ["seed", "frame_stamp", "epoch", "sub_experiment_path", "experiment_name"]:
        if col in best_row.index:
            out[f"best_{col}"] = best_row[col]

    # include hyperparams from that best row
    for c in hyperparam_columns:
        out[c] = best_row[c]

    return out


rows = []
for sub_dir in experiment_sub_dirs:
    exp_path = os.path.join(results_root, sub_dir)
    if not os.path.isdir(exp_path):
        print(f"[WARN] Not found: {exp_path}")
        continue

    df = process_experiment(exp_path)
    best = extract_best_peak_validation(df, metric=metric)
    best["experiment_sub_dir"] = sub_dir
    rows.append(best)

compare_df = pd.DataFrame(rows)

if len(compare_df) == 0:
    compare_df
else:
    preferred = [
        "experiment_sub_dir",
        "best_peak_reward",
        "best_seed",
        "best_epoch",
        "best_frame_stamp",
        "best_sub_experiment_path",
        "best_experiment_name",
    ]
    cols = [c for c in preferred if c in compare_df.columns] + [c for c in compare_df.columns if c not in preferred]
    compare_df = compare_df[cols].sort_values("best_peak_reward", ascending=False).reset_index(drop=True)
    compare_df


In [6]:
compare_df

,experiment_sub_dir,best_peak_reward,best_seed,best_frame_stamp,best_sub_experiment_path,best_experiment_name,sub_exp_cfg_optim.args_.lr,sub_exp_cfg_agent_params.args_.target_soft_tau,sub_exp_cfg_experiment
0,2025Oct28-115156_configs,-7.996198,0,5400000,d:\Work\repos\RL\phd-rl-algos\dqn\opinion_dyna...,0001_optim.args_.lr_1e-05__agent_params.args_....,1e-05,0.0005,fixed_res
1,2025Oct30-234652_configs,-8.243773,1,7000000,d:\Work\repos\RL\phd-rl-algos\dqn\opinion_dyna...,0002_optim.args_.lr_3e-05__agent_params.args_....,3e-05,0.0001,fixed_res
2,2025Oct29-151336_configs,-9.546037,0,6400000,d:\Work\repos\RL\phd-rl-algos\dqn\opinion_dyna...,0000_optim.args_.lr_1e-05__agent_params.args_....,1e-05,0.0001,fixed_res
3,2025Oct31-115130_configs,-13.657468,0,5000000,d:\Work\repos\RL\phd-rl-algos\dqn\opinion_dyna...,0002_optim.args_.lr_3e-05__agent_params.args_....,3e-05,0.0001,fixed_res
